# முன்னுரிமை உறுப்பினர் மிடில்வேர் உடன் ஹோட்டல் முன்பதிவு

இந்த நோட்புக் Microsoft ஏஜெண்ட் கட்டமைப்பைப் பயன்படுத்தி **விளக்க செயல்பாட்டு மிடில்வேர்** ஐ காட்டுகிறது. நாங்கள் நிபந்தனையும் கொண்ட வேலைப்ப fluxo எடுத்துக்காட்டில் முன்னுரிமை உறுப்பினர்களுக்கு சிறப்பு உரிமைகளை வழங்கும் ஒரு மிடில்வேர் அடுக்கு ஒன்றை சேர்க்கிறோம்.

## நீங்கள் கற்றுக் கொள்வதைக்:
1. **விளக்க செயல்பாட்டு மிடில்வேர்**: செயல்பாட்டின் முடிவுகளை இடைமறித்து மாற்றுதல்
2. **சூழல் அணுகல்**: செயல்பாடு முடிந்த பிறகு `context.result` ஐ வாசித்து, மாற்றுதல்
3. **வணிக தாதத்தின் செயலாக்கம்**: முன்னுரிமை உறுப்பினர் நன்மைகள்
4. **முடிவு மாற்றல்**: பயனர் நிலையைப் பொறுத்து செயல்பாட்டின் முடிவுகளை மாற்றுதல்
5. **அதே வேலைப்ப fluxo, வெவ்வேறு முடிவுகள்**: மிடில்வேர் மூலம் நடந்து கொண்ட நடத்தை மாற்றங்கள்

## மிடில்வேர் உடன் வேலைப்ப fluxo கட்டமைப்பு:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## நிபந்தனை வேலைப்ப fluxo உடன் முக்கிய வேறுபாடு:

**மிடில்வேர் இல்லாமல்** (14-conditional-workflow.ipynb):
- பாரிஸ் நகரில் அறைகள் இல்லை → alternative_agentக்கு வழிமாற்று

**மிடில்வேர் உடன்** (இந்த நோட்புக்):
- சாதாரண பயனர் + பாரிஸ் → அறைகள் இல்லை → alternative_agentக்கு வழிமாற்று
- முன்னுரிமை பயனர் + பாரிஸ் → 🌟 மிடில்வேர் மாற்றுகிறது! → கிடைக்கும் → booking_agentக்குச் செல்லும்

## ஆரம்பிக்க தேவையானவை:
- Microsoft ஏஜெண்ட் கட்டமைப்பு நிறுவப்பட்டது
- நிபந்தனையும் கொண்ட வேலைப்ப fluxo பற்றிய புரிதல் (பார்க்கவும் 14-conditional-workflow.ipynb)
- GitHub டோக்கன் அல்லது OpenAI API விசை
- மிடில்வேர் மாதிரிகள் பற்றிய அடிப்படை அறிவு


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## அடுக்கடி 1: அமைப்பான வெளியீடுகளுக்கு Pydantic மாதிரிகள் வரையறுக்கவும்

இந்த மாதிரிகள் முகவர்கள் திருப்பி அளிக்கும் **திட்டக்கோவையை** வரையறுக்கின்றன. மிடில் வெலர் கிடைக்கும் முடிவை மாற்றும் போது `priority_override` என்ற புலத்தையும் சேர்த்துள்ளோம்.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## படி 2: முன்னுரிமை உறுப்பினர்கள் தரவுத்தளம் வரையறு

இந்த டெமோக்காக, நாம் முன்னுரிமை உறுப்பினர் தரவுத்தளத்தை சிமுலேட் செய்வோம். உற்பத்தியில், இது ஒரு உண்மையான தரவுத்தள அல்லது API-யை கேள்வி செய்வது.

**முன்னுரிமை உறுப்பினர்கள்:**
- `alice@example.com` - VIP உறுப்பினர்
- `bob@example.com` - பிரீமியம் உறுப்பினர்  
- `priority_user` - சோதனை கணக்கு


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## படி 3: ஹோட்டல் புக்கிங் கருவியை உருவாக்கவும்

நிபந்தனை ஒருங்கிணைப்புப் பணியோடு அதேபோல், ஆனால் இப்போது இது மிடில்‌வேர் மூலம் தடுக்கப்படும்!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## படி 4: 🌟 முன்னுரிமை சரிபார்ப்புப் படிமுறையை உருவாக்கவும் (முக்கிய அம்சம்!)

இது இந்த நோட்புக் இன் **முக்கிய செயல்பாடு** ஆகும். மிடில் வெர்:

1. **hotel_booking** செயல்பாட்டு அழைப்பை இடைமறிக்கிறது
2. `next(context)` ஐ அழைத்து சாதாரணமாக செயல்பாட்டை **நடத்தியது**
3. `context.result` இல் முடிவை **பரிசீலிக்கிறது**
4. பயனர் முன்னுரிமையுடையவராகவும் அறைகள் கிடைக்காவிடிலும்கூட முடிவை **மேம்படுத்துகிறது**
5. மாற்றப்பட்ட முடிவை ஏஜென்டிற்கு **திருப்பிச் செலுத்துகிறது**

**முக்கிய கட்டமைப்பு:**
```python
async def my_middleware(context, next):
    await next(context)  # செயலை இயக்கவும்
    # இப்போது context.result செயல்பாட்டின் வெளியீட்டை கொண்டுள்ளது
    if some_condition:
        context.result = new_value  # மேலோட்டமாக்கவும்!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## படி 5: வழித்தடத்திற்கான நிபந்தனை செயல்பாடுகளை வரையறு

நிபந்தனை பணிச்சூழலைப்போல் அதே நிபந்தனை செயல்பாடுகள் - அவை ஒழுங்குபடுத்தப்பட்ட வெளியீட்டை ஆய்வு செய்து வழித்தடத்தை நிர்ணயிக்கும்.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## படி 6: தனிப்பயன் காட்சிப் பணியாளரை உருவாக்கவும்

முன்பு இருந்த அதே பணியாளர் - இறுதி வேலைநிறுத்தத்தின் வெளியீடு காண்பிக்கிறது.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## படி 7: சுற்றுச்சூழல் மாறிகள் ஏற்றுகொள்ளவும்

LLM கிளையண்டை (Microsoft Foundry / Azure OpenAI) கட்டமைக்கவும்.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## படி 8: மிடில்வேருடன் AI முகவர்களை உருவாக்கவும்

**முக்கிய வேறுபாடு:** availability_agent ஐ உருவாக்கும்போது, நாம் `middleware` அளவுருவை அனுப்புகிறோம்!

இது எப்படி priority_check_middleware ஐ முகவரின் செயல்பாட்டுக் கூப்பலுக்கு ஊற்றுகிறோம் என்பதைக் காண்பிக்கிறது.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## படி 9: வேலைவ sérieை கட்டமைக்கவும்

முந்தைய வேலைவ sérieை கட்டமைப்புடன் ஒரே மாதிரி - கிடைக்கும் நிலையைப் பொறுத்து நிபந்தனையான வழிசெலுத்தல்.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## படி 10: சோதனை வழக்கு 1 - பாரிஸில் சாதாரண பயனர் (ஒழுங்குபடுத்தல் இல்லை)

ஒரு சாதாரண பயனர் பாரிஸ் முன்பதிவு செய்ய முயற்சிக்கிறார் → அறைகள் கிடையாது → alternative_agent க்கு வழிசெலுத்தப்படுகிறது


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## படி 11: சோதனை வழக்கு 2 - 🌟 பாரிஸில் முக்கியத்துவம் வாய்ந்த பயனர் (மீள்பரிசீலனையுடன்!)

ஒரு முக்கிய உறுப்பினர் பாரிஸை முன்பதிவு செய்ய முயற்சிக்கிறார் → முதலில் அறைகள் இல்லை → 🌟 மிடில்்வேர் மீள்பரிசீலனைகள்! → booking_agent க்கு வழிமாற்றம்

**இது மிடில் வேர் அதிகாரத்தின் முக்கிய காட்சி!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## படி 12: சோதனை வழக்கு 3 - ஸ்டாக்ஹோலம் நகரில் முன்னுரிமை பயனர் (ஏற்கனவே உள்ளடக்கம்)

முன்னுரிமை பயனர் ஸ்டாக்ஹோலம் → அறைகள் கிடைக்கிறது → மாற்றம் தேவையில்லை → booking_agent க்கு வழிமுறைகள்

இது காட்டுகிறது மிடில்வேர் தேவையானபோது மட்டும் செயல்படுகிறது!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## முக்கியமான கருத்துகள் மற்றும் மிடில்வேர் கருத்துக்கள்

### ✅ நீங்கள் கற்றுக்கொண்டவை:

#### **1. செயல்பாடு-அடிப்படையிலான மிடில்வேர் படிமுறை**

மிடில்வேர் எளிய அசிங்க் செயல்பாட்டை பயன்படுத்தி செயல்பாட்டு அழைப்புகளை தடுக்கிறது:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # செயல்பாடு இயக்கத்திற்கு முன்னர்
    print("Intercepting...")
    
    # செயல்பாட்டை இயக்கவும்
    await next(context)
    
    # செயல்பாடு இயக்கத்துக்குப் பிறகு - முடிவை பரிசீலிக்கவும்
    if context.result:
        # தேவைப்பட்டால் முடிவை மாற்றவும்
        context.result = modified_value
```

#### **2. சூழல் அணுகல் மற்றும் முடிவு மீறல்**

- `context.function` - அழைக்கப்படும் செயல்பாட்டை அணுகுதல்
- `context.arguments` - செயல்பாட்டு அ argumentos படிக்கவும்
- `context.kwargs` - கூடுதல் அளவுருக்களை அணுகுதல்
- `await next(context)` - செயல்பாட்டை இயக்கவும்
- `context.result` - செயல்பாட்டின் பயன்வளத்தை படிக்கவும்/மாற்றவும்

#### **3. வணிக தர்க்க நடைமுறை**

எங்கள் மிடில்வேர் முன்னுரிமை உறுப்பினர் நன்மைகளை செயல்படுத்துகிறது:
- **பொது பயனர்கள்**: மாற்றங்கள் வேண்டாம், வழக்கமான வேலைப்பாதை
- **முன்னுரிமை பயனர்கள்**: "கிடைக்கவில்லை" ஐ "கிடைக்கிறது" என்று மாற்றுதல்
- **நிலையான தர்க்கம்**: தேவையான நேரத்தில் மட்டும் மாற்றம் செய்கிறது

#### **4. அதே வேலைப்பாதை, வேறுபட்ட முடிவுகள்**

மிடில்வேர் சக்தி:
- ✅ வேலைப்பாதை அமைப்பில் மாற்றங்கள் இல்லாமல்
- ✅ கருவி செயல்பாட்டில் மாற்றங்கள் இல்லாமல்
- ✅ நிலையான வழிசெலுத்தல் தர்க்கத்தில் மாற்றங்கள் இல்லாமல்
- ✅ வெறும் மிடில்வேர் → வேறுபட்ட நடத்தைகள்!

### 🚀 நிஜ உலக பயன்பாடுகள்:

1. **VIP/பிரீமியம் அம்சங்கள்**
   - பிரீமியம் பயனர்களுக்கான விகித வரம்புகளை மீறல்
   - வளங்களுக்கு முன்னுரிமை அணுகல் வழங்கல்
   - பிரீமியம் அம்சங்களை தானாக திறத்தல்

2. **A/B சோதனை**
   - பயனர்களை பல்வேறு செயலாக்கங்களுக்கு வழிமறித்தல்
   - குறிப்பிட்ட பயனர்களோடு புதிய அம்சங்களை சோதனை செய்தல்
   - படிப்படியாக அம்சங்களை வெளியீடு செய்தல்

3. **பாதுகாப்பு & விதிமுறைகள் பின்பற்றல்**
   - செயல்பாட்டு அழைப்புகளை சரிபார்த்து
   - நுண்ணறிவு நடவடிக்கைகளை தடுப்பு
   - வணிக விதிகளை கடைபிடித்தல்

4. **செயல்திறன் மேம்பாடு**
   - குறிப்பிட்ட பயனர்களுக்கான முடிவுகளை சேமித்தல்
   - சாத்தியமானபோது செலவான செயல்களை தவிர்க்கல்
   - தானாக வள ஒதுக்கீடு

5. **பிழை கையாளல் & மறுஆய்வு**
   - பிழைகளை கவனித்து மென்மையாக கையாளல்
   - மறுஆய்வு தர்க்கத்தை செயல்படுத்தல்
   - மாற்றுவழி செயலாக்கங்களுக்கு fallback

6. **பதிவு & கண்காணித்தல்**
   - செயல்பாட்டு செயலாக்க நேரங்களை கண்காணித்தல்
   - அளவுருக்கள் மற்றும் முடிவுகளை பதிவு செய்தல்
   - பயன்படுத்தும் தழுவல் முறைகளை கண்காணித்தல்

### 🔑 அலங்காரிகளோடு வித்தியாசங்கள்:

| அம்சம் | அலங்காரி | மிடில்வேர் |
|---------|-----------|------------|
| ** பரபரப்பு ** | ஒரு செயல்பாடு | முகவர் உள்ள அனைத்து செயல்பாடுகளும் |
| **பல்திறன்** | வரையறுப்பில் நிர்ணயிக்கப்பட்டது | இயக்க நேரத்தில் மாற்றக்கூடியது |
| **சூழல்** | கட்டுப்படுத்தப்பட்டது | முழு முகவர் சூழல் |
| **இணைப்பு** | பல அலங்காரிகள் | மிடில்வேர் குழாய் |
| **முகவர் அறிவு** | இல்லை | ஆம் (முகவர் வகையை அணுகுதல்) |

### 📚 எப்போது மிடில்வேர் பயன்படுத்த வேண்டும்:

✅ **பயன்பாடு என்னால்:**
- பயனர்/மூலம் நிலையைக் கொண்டு நடத்தையை மாற்ற வேண்டிய நேரத்தில்
- பல செயல்பாடுகளுக்கு தர்க்கம் பொருத்த வேண்டும்போது
- முகவர்-நிலை சூழல் அணுகல் தேவையான நேரத்தில்
- குறுக்கு கவலைகளை (பதிவு, அங்கீகாரம், முதலியன) செயல்படுத்தும்போது

❌ **மிடில்வேர் பயன்படுத்த வேண்டாம்:**
- எளிய உள்ளீடு சான்று (Pydantic பயன்படுத்தவும்)
- செயல்பாடு-தனிப்பட்ட தர்க்கம் (செயல்பாட்டில் வைத்திருங்கள்)
- ஒருமுறை மாற்றங்கள் (செயல்பாட்டை நேரடியாக மாற்றுங்கள்)

### 🎓 மேம்பட்ட படிமுறைகள்:

```python
# பல மிடில்வேர் (நிர்வகிப்பு வரிசை முக்கியம்!)
middleware=[
    logging_middleware,      # முதலில் பதிவுகள்
    auth_middleware,         # பின்னர் அங்கீகாரம் சோதனை
    cache_middleware,        # பின்னர் கேச்சை சோதனை
    rate_limit_middleware,   # பின்னர் விகிதம் வரம்புகள்
    priority_check_middleware  # கடைசியாக முன்னுரிமை சோதனை
]

# நிபந்தனை அடிப்படையிலான மிடில்வேர் செயலாக்கம்
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # முடிவை மாற்று
    else:
        # செயலாக்கத்தை முழுக்க தவிர்க்கவும்
        context.result = cached_value
```

### 🔗 தொடர்புடைய கருத்துக்கள்:

- **முகவர் மிடில்வேர்**: agent.run() அழைப்புகளை தடுக்கிறது
- **செயல்பாடு மிடில்வேர்**: கருவி செயல்பாட்டு அழைப்புகளை தடுக்கிறது (நாம் பயன்படுத்தியது!)
- **மிடில்வேர் குழாய்**: மிடில்வேர் வரிசையில் கட்டமைக்கப்பட்டுள்ளது
- **சூழல் பரப்புதல்**: மிடில்வேர் வரிசையில் நிலையைப் பறத்தல்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
